# `bls(series, start, end=None)` and `bls_series()`

BLS labor and price series from `indicators.bls`, one value per month, oldest first. `bls_series()` is the map from each series id to its name, units, frequency and category. Every cell runs against prod, read-only, as `readonly_user`. Change the arguments and re-run any cell.

A series is named by its BLS id or by its name, in any case and spacing. Months with no value are left out, never filled.

In [1]:
from datetime import date, timedelta

import polars as pl
from IPython.display import Markdown

from arora_data.bls import BlsSeries, BlsValue
from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.sync import bls, bls_series


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## The series
Every series the table holds, read from it on every call. The labels come from each series' latest row; `first_date` and `latest_date` are its first and latest months with a value. Values are as BLS publishes them, in these units: the unemployment rate is percent (4.4 is 4.4%), payrolls and job openings are thousands, CPI-U and PPI are index levels.

In [2]:
show(bls_series())

5 rows

series_id|dataset|name|units|frequency|category|first_date|latest_date
---|---|---|---|---|---|---|---
CES0000000001|bls/core|Total nonfarm employment (SA)|thousands of persons|monthly|Labor|1939-01-01|2026-10-01
CUUR0000SA0|bls/core|CPI-U, all items (NSA)|index 1982-84=100|monthly|Inflation|1913-01-01|2026-10-01
JTS000000000000000JOL|bls/core|Job openings, JOLTS (SA)|thousands|monthly|Labor|2000-12-01|2026-09-01
LNS14000000|bls/core|Unemployment rate (SA)|percent|monthly|Labor|1948-01-01|2026-10-01
WPSFD4|bls/core|PPI, final demand|index|monthly|Inflation|2009-11-01|2026-09-01

## The map between names and ids
Two plain dicts from the same call. Every name is unique, so either one looks a series up.

In [3]:
by_name = {s.name: s.series_id for s in bls_series()}
by_id = {s.series_id: s.name for s in bls_series()}
assert len(by_name) == len(by_id)
print(by_name["Unemployment rate (SA)"], "|", by_id["JTS000000000000000JOL"])

LNS14000000 | Job openings, JOLTS (SA)


## One series, by id or by name
The unemployment rate over the last twelve months. The id and the name, in any case and spacing, give the same rows.

In [4]:
year = today_et() - timedelta(days=365)
assert bls("LNS14000000", year) == bls("lns14000000", year) == bls("  unemployment RATE (sa) ", year)
show(bls("LNS14000000", year))

11 rows

series_id|name|date|value|units
---|---|---|---|---
LNS14000000|Unemployment rate (SA)|2025-12-01|4.40000000|percent
LNS14000000|Unemployment rate (SA)|2026-01-01|4.30000000|percent
LNS14000000|Unemployment rate (SA)|2026-02-01|4.40000000|percent
LNS14000000|Unemployment rate (SA)|2026-03-01|4.30000000|percent
LNS14000000|Unemployment rate (SA)|2026-04-01|4.30000000|percent
LNS14000000|Unemployment rate (SA)|2026-05-01|4.30000000|percent
LNS14000000|Unemployment rate (SA)|2026-06-01|4.20000000|percent
LNS14000000|Unemployment rate (SA)|2026-07-01|4.10000000|percent
LNS14000000|Unemployment rate (SA)|2026-08-01|4.10000000|percent
LNS14000000|Unemployment rate (SA)|2026-09-01|4.20000000|percent
LNS14000000|Unemployment rate (SA)|2026-10-01|4.10000000|percent

## Gaps are left out
BLS never published October 2025 for the unemployment rate or CPI-U, because of the shutdown, so both skip 2025-10-01. Payrolls, job openings and PPI have it.

In [5]:
for s in bls_series():
    months = [v.date for v in bls(s.series_id, date(2025, 8, 1), date(2025, 12, 31))]
    print(f"{s.series_id:<22} has 2025-10-01: {date(2025, 10, 1) in months}")

CES0000000001          has 2025-10-01: True


CUUR0000SA0            has 2025-10-01: False


JTS000000000000000JOL  has 2025-10-01: True


LNS14000000            has 2025-10-01: False


WPSFD4                 has 2025-10-01: True


## How dates are set
Each month is dated at its first day, and BLS's annual averages are not stored. A month appears once BLS publishes it, a few weeks after it ends, so the series end at different months; the second line shows each one's latest.

In [6]:
print([v.date for v in bls("CES0000000001", date(2026, 5, 1))])
print({s.series_id: str(s.latest_date) for s in bls_series()})

[datetime.date(2026, 5, 1), datetime.date(2026, 6, 1), datetime.date(2026, 7, 1), datetime.date(2026, 8, 1), datetime.date(2026, 9, 1), datetime.date(2026, 10, 1)]


{'CES0000000001': '2026-10-01', 'CUUR0000SA0': '2026-10-01', 'JTS000000000000000JOL': '2026-09-01', 'LNS14000000': '2026-10-01', 'WPSFD4': '2026-09-01'}


## Full history in one call
From 1900 there is no row limit; a series starts at its own first value. The unemployment rate from 1948, with each decade's lowest and highest month.

In [7]:
history = to_df(bls("LNS14000000", date(1900, 1, 1)), BlsValue)
print(f"{history.height:,} months, {history['date'].min()} to {history['date'].max()}")
history.group_by(pl.col("date").dt.year() // 10 * 10).agg(pl.col("value").min().alias("lowest, %"), pl.col("value").max().alias("highest, %")).sort("date").rename({"date": "decade"})

945 months, 1948-01-01 to 2026-10-01


decade,"lowest, %","highest, %"
i32,"decimal[38,8]","decimal[38,8]"
1940,3.40000000,7.90000000
1950,2.50000000,7.50000000
1960,3.40000000,7.10000000
1970,3.90000000,9.00000000
1980,5.00000000,10.80000000
1990,4.00000000,7.80000000
2000,3.80000000,10.00000000
2010,3.50000000,9.90000000
2020,3.40000000,14.80000000


## Not here
Seasonally adjusted CPI is `fred("CPIAUCSL")`, and U-6 unemployment is `fred("U6RATE")`. GDP is `bea("A191RC")`.

## Bad input
An unknown id or name, or a start after the end, raises a clear error before any values are read.

In [8]:
for args in [("CPIAUCSL", date(2026, 1, 1)), ("unemployment rate", date(2026, 1, 1)), (None, date(2026, 1, 1)),
             ("LNS14000000", date(2026, 1, 2), date(2026, 1, 1))]:
    try:
        bls(*args)
    except ValueError as e:
        print(e)

'CPIAUCSL' is not a stored BLS series: use an id or name from bls_series(), such as LNS14000000 or 'Unemployment rate (SA)'


'unemployment rate' is not a stored BLS series: use an id or name from bls_series(), such as LNS14000000 or 'Unemployment rate (SA)'


None is not a stored BLS series: use an id or name from bls_series(), such as LNS14000000 or 'Unemployment rate (SA)'
start must be on or before end
